# This should help us create the required data files 
We have decided to train the model on the data from the years 2022 and 2023, then test on the data from 2024.

In [1]:
# importing the required libraries
import pandas as pd

In [2]:
# original file path
path_to_data = '../data/raw/Intuilize_MNSU_ACME_SalesData.csv'

In [3]:
# converting csv to dataframe
data = pd.read_csv(path_to_data)

In [4]:
# Convert the date column to datetime format and adding month and year
data['SalesDate'] = pd.to_datetime(data['SalesDate'])
data["Month"] = data["SalesDate"].dt.month
data["Year"] = data["SalesDate"].dt.year

# Calculate profit
data["Profit"] = data["ExtPrice"] - data["ExtCost"]
data['ProfitMargin'] = data['Profit'] / data['ExtPrice'].replace(0, 1)
data['ProfitCost_Ratio'] = data['UnitPrice'] / data['UnitCost'].replace(0, 1)

removing 2021 and 2025 data as they are incomplete

In [5]:
data = data.drop(data[data["Year"].isin([2021, 2025])].index)

In [6]:
# creating customer aliases
unique_customers = data['CustomerName'].unique()
name_to_alias = {name: f"Customer {i+1}" for i, name in enumerate(unique_customers)}
data['customer_alias'] = data['CustomerName'].map(name_to_alias)

In [7]:
_2022_and_2023_data = data[data["Year"].isin([2022, 2023])]
_2024_data = data[data["Year"] == 2024]

# train_data.csv contains all the data from 2022 and 2023
# test_data.csv contains all the data from 2024

In [8]:
# Aggregate revenue, profit, and frequency per customer while ensuring CustomerID is retained
customer_metrics = data.groupby(["CustomerID", "CustomerName"]).agg(
    TotalRevenue=("ExtPrice", "sum"),
    TotalProfit=("Profit", "sum"),
    PurchaseFrequency=("OrderNumber", "nunique"),
).reset_index()

Save data for convenience

In [9]:
_2022_and_2023_data.to_csv("../data/raw/train_data.csv")
_2024_data.to_csv("../data/raw/test_data.csv")
customer_metrics.to_csv("../data/raw/customer_metrics.csv")
data.to_csv("../data/raw/raw_data_extra_features.csv") # removed 2021 and 2025 and have profit, month, and year